In [0]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [0]:
%sql
SELECT current_catalog();

In [0]:
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# base_volume
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

f = "restaurant_reviews.csv"

dbutils.fs.cp(f"file:/Workspace/Users/darsh2115@gmail.com/demo/GenAI/data/{f}", f"{volume_base}/{f}")

# Creating the Bronze layer dataframe

In [0]:
from delta.tables import *
import pyspark.sql.functions as F

In [0]:
bronze_df = spark.read.format("csv").option("header", "true").load(f"/Volumes/{catalog_name}/default/genai_lab/restaurant_reviews.csv")

bronzeDeltaTablePath = f"/Volumes/{catalog_name}/default/genai_lab/bronze_restaurant_reviews"
bronze_df.write.format("delta").mode("overwrite").save(bronzeDeltaTablePath)


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo.Bronze

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("Bronze.bronze_restaurant_reviews")




# Creating Gold Layer

In [0]:
gold_df = spark.read.format("delta").table("Bronze.bronze_restaurant_reviews")

gold_df.display()

In [0]:
gold_df.count()

In [0]:
gold_df = spark.sql("""
SELECT
    CustomerID,
    CustomerName,                     
    Products,
    Review,
    ai_classify(
        Review,
        '{"Positive": "Overall favorable or happy sentiment", "Neutral": "No strong sentiment either way", "Negative": "Overall unfavorable or unhappy sentiment"}',
        MAP('version', '2.0', 'instructions', 'Classify the customer sentiment of this restaurant review.')
    ):response[0]::STRING AS sentiment,
    ai_query(
        "databricks-meta-llama-3-3-70b-instruct",
        "Summarise this restaurant review in one short sentence: " || Review,
        returnType => 'STRING'
    ) AS summary
FROM demo.Bronze.bronze_restaurant_reviews               
""")

In [0]:
gold_df.display()

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo.Gold;

In [0]:

from delta.tables import *
from pyspark.sql.functions import *

# Storing the Gold Layer as a Delta Table in the Databricks File System (DBFS)
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/genai_lab/gold/gold_restaurant_reviews'
gold_df.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Storing the Gold Layer as a Delta Table in the Data Catalog
gold_df.write.format('delta').mode("overwrite").saveAsTable('Gold.restaurant_reviews')